# 🗑️ Waste Classifier — Exploratory Data Analysis
### Notebook 02 | Preprocessing & EDA

This notebook explores the dataset before any model training.

**Goals:**
- Visualize class distribution
- Inspect sample images per class
- Compute average image per class (visual fingerprint)
- Analyze image size distribution
- Save all plots to `results/`

In [ ]:
import sys
sys.path.append(".")
from utils import *

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict

# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

print("✅ Imports done")

## 📁 Dataset Paths

In [ ]:
# Root paths
DATA_DIR = "../data/processed"
RESULTS_DIR  = "../results"
CLASS_NAMES  = ["Glass", "Metal", "Organic", "Paper", "Plastic"]

os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Data directory : {DATA_DIR}")
print(f"Results directory : {RESULTS_DIR}")
print(f"Classes : {CLASS_NAMES}")

## 📊 Class Distribution

In [ ]:
# Count images per class per split
split_counts = {}

for split in ["train", "val", "test"]:
    split_counts[split] = {}
    for cls in CLASS_NAMES:
        cls_path = os.path.join(DATA_DIR, split, cls)
        split_counts[split][cls] = len(os.listdir(cls_path))

# Print summary table
print(f"{'Class':<12}", end="")
for split in ["train", "val", "test"]:
    print(f"{split:>8}", end="")
print(f"{'Total':>8}")
print("-" * 44)

for cls in CLASS_NAMES:
    total = sum(split_counts[split][cls] for split in ["train", "val", "test"])
    print(f"{cls:<12}", end="")
    for split in ["train", "val", "test"]:
        print(f"{split_counts[split][cls]:>8}", end="")
    print(f"{total:>8}")

print("-" * 44)
grand_total = sum(split_counts[s][c] for s in ["train","val","test"] for c in CLASS_NAMES)
print(f"{'TOTAL':<12}", end="")
for split in ["train", "val", "test"]:
    print(f"{sum(split_counts[split].values()):>8}", end="")
print(f"{grand_total:>8}")

## 📊 Class Distribution Bar Chart

In [ ]:
# Bar chart — class distribution across splits
splits = ["train", "val", "test"]
colors = ["steelblue", "tomato", "seagreen"]

x = np.arange(len(CLASS_NAMES))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))

for i, (split, color) in enumerate(zip(splits, colors)):
    counts = [split_counts[split][cls] for cls in CLASS_NAMES]
    bars = ax.bar(x + i * width, counts, width, label=split.capitalize(), color=color)
    ax.bar_label(bars, padding=3, fontsize=9)

ax.set_xlabel("Class", fontsize=12)
ax.set_ylabel("Number of Images", fontsize=12)
ax.set_title("Class Distribution Across Splits", fontsize=14)
ax.set_xticks(x + width)
ax.set_xticklabels(CLASS_NAMES)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "class_distribution.png"), dpi=150)
plt.show()
print("✅ Saved → results/class_distribution.png")

## 🖼️ Sample Images Per Class

In [ ]:
# Display 5 sample images per class from train set
n_samples = 5
fig, axes = plt.subplots(len(CLASS_NAMES), n_samples, figsize=(15, 12))

for i, cls in enumerate(CLASS_NAMES):
    cls_path = os.path.join(DATA_DIR, "train", cls)
    images   = os.listdir(cls_path)[:n_samples]
    for j, img_name in enumerate(images):
        img = Image.open(os.path.join(cls_path, img_name)).convert("RGB")
        axes[i, j].imshow(img)
        axes[i, j].axis("off")
        if j == 0:
            axes[i, j].set_title(cls, fontsize=13, fontweight="bold", loc="left")

plt.suptitle("Sample Images Per Class", fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "sample_images.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → results/sample_images.png")

## 🎨 Average Image Per Class (Visual Fingerprint)

In [ ]:
# Compute average image per class (mean of all pixel values)
fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(15, 4))

for i, cls in enumerate(CLASS_NAMES):
    cls_path   = os.path.join(DATA_DIR, "train", cls)
    img_files  = os.listdir(cls_path)
    avg_img    = np.zeros((224, 224, 3), dtype=np.float32)

    for img_name in img_files:
        img = Image.open(os.path.join(cls_path, img_name)).convert("RGB")
        img = img.resize((224, 224))
        avg_img += np.array(img, dtype=np.float32)

    avg_img /= len(img_files)
    avg_img  = avg_img.astype(np.uint8)

    axes[i].imshow(avg_img)
    axes[i].set_title(cls, fontsize=12, fontweight="bold")
    axes[i].axis("off")

plt.suptitle("Average Image Per Class (Visual Fingerprint)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "average_images.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → results/average_images.png")

## 📐 Image Size Distribution

In [ ]:
# Collect image dimensions from train set
widths, heights = [], []

for cls in CLASS_NAMES:
    cls_path = os.path.join(DATA_DIR, "train", cls)
    for img_name in os.listdir(cls_path):
        img = Image.open(os.path.join(cls_path, img_name))
        w, h = img.size
        widths.append(w)
        heights.append(h)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(widths,  bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("Width Distribution")
axes[0].set_xlabel("Width (px)")
axes[0].set_ylabel("Count")
axes[0].axvline(np.mean(widths), color="tomato", linestyle="--", label=f"Mean: {np.mean(widths):.0f}px")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(heights, bins=40, color="seagreen", edgecolor="white")
axes[1].set_title("Height Distribution")
axes[1].set_xlabel("Height (px)")
axes[1].set_ylabel("Count")
axes[1].axvline(np.mean(heights), color="tomato", linestyle="--", label=f"Mean: {np.mean(heights):.0f}px")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("Image Size Distribution (Train Set)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "size_distribution.png"), dpi=150)
plt.show()
print(f"✅ Saved → results/size_distribution.png")
print(f"   Width  — min: {min(widths)}  max: {max(widths)}  mean: {np.mean(widths):.0f}")
print(f"   Height — min: {min(heights)}  max: {max(heights)}  mean: {np.mean(heights):.0f}")

## 🌈 Pixel Intensity Distribution Per Class

In [ ]:
# Average pixel intensity histogram per class (R, G, B channels)
fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(18, 4), sharey=True)
channel_colors = ["red", "green", "blue"]

for i, cls in enumerate(CLASS_NAMES):
    cls_path  = os.path.join(DATA_DIR, "train", cls)
    img_files = os.listdir(cls_path)[:100]  # sample 100 for speed
    pixels    = []

    for img_name in img_files:
        img = Image.open(os.path.join(cls_path, img_name)).convert("RGB")
        img = img.resize((64, 64))
        pixels.append(np.array(img))

    pixels = np.stack(pixels)  # (N, 64, 64, 3)

    for c, color in enumerate(channel_colors):
        axes[i].hist(
            pixels[:, :, :, c].flatten(),
            bins=50, alpha=0.5, color=color,
            label=color.upper(), density=True
        )

    axes[i].set_title(cls, fontsize=11, fontweight="bold")
    axes[i].set_xlabel("Pixel Value")
    if i == 0:
        axes[i].set_ylabel("Density")
    axes[i].legend(fontsize=7)
    axes[i].grid(alpha=0.3)

plt.suptitle("Pixel Intensity Distribution Per Class (RGB)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "pixel_intensity.png"), dpi=150)
plt.show()
print("✅ Saved → results/pixel_intensity.png")

## ✅ EDA Summary

In [ ]:
print("=" * 50)
print("        EDA COMPLETE — Summary")
print("=" * 50)
print(f"  Total images     : {grand_total}")
print(f"  Classes          : {CLASS_NAMES}")
print(f"  Train / Val / Test split: 70 / 15 / 15")
print()
print("  Plots saved to results/:")
print("    ✅ class_distribution.png")
print("    ✅ sample_images.png")
print("    ✅ average_images.png")
print("    ✅ size_distribution.png")
print("    ✅ pixel_intensity.png")
print()
print("=" * 50)